In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load local CSV file (Semicolon ';' separated)
df = pd.read_csv('winequality-red.csv', sep=';')

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
df.head()

In [ ]:
# Check missing values and data types
print("Missing Values Check:")
print(df.isnull().sum())

# Class Distribution of Target Variable 'quality'
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='quality', palette='viridis')
plt.title('Distribution of Wine Quality Scores (3-8)', fontsize=14, fontweight='bold')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.show()

# Display value counts
print("\nQuality Score Counts:")
print(df['quality'].value_counts().sort_index())

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Chemical Features', fontsize=14, fontweight='bold')
plt.show()

# Key Feature Relationships with Quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alcohol vs Quality
sns.boxplot(data=df, x='quality', y='alcohol', ax=axes[0], palette='Blues')
axes[0].set_title('Alcohol Level vs Wine Quality', fontsize=12, fontweight='bold')

# Volatile Acidity vs Quality
sns.boxplot(data=df, x='quality', y='volatile acidity', ax=axes[1], palette='Reds')
axes[1].set_title('Volatile Acidity vs Wine Quality', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Binning Quality into Binary Target (Good: >= 7, Bad/Average: < 7)
df['quality_label'] = df['quality'].apply(lambda x: 1 if x >= 7 else 0)

print("Binary Class Distribution (0 = Bad/Average, 1 = Good):")
print(df['quality_label'].value_counts())

# Features and Target
X = df.drop(columns=['quality', 'quality_label'])
y = df['quality_label']

# 2. Stratified Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Feature Scaling (Essential for SVC and SGD)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set size: {X_train.shape[0]} | Testing set size: {X_test.shape[0]}")

In [ ]:
# Initialize Classifiers
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Stochastic Gradient Descent (SGD)": SGDClassifier(max_iter=1000, random_state=42),
    "Support Vector Classifier (SVC)": SVC(kernel='rbf', C=1.0, random_state=42)
}

results = {}

# Train and Evaluate Models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, model) in enumerate(models.items()):
    # Tree models don't require scaling, but distance/gradient models do
    if name == "Random Forest":
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    
    # Confusion Matrix Visualization
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{name}\nAccuracy: {acc:.2%}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
for name, model in models.items():
    preds = model.predict(X_test if name == "Random Forest" else X_test_scaled)
    print(f"==================== {name} ====================")
    print(classification_report(y_test, preds, target_names=['Bad/Avg (0)', 'Good (1)']))

In [ ]:
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='teal')
plt.title('Random Forest Feature Importances', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Chemical Feature')
plt.show()

In [ ]:
# Comparison Summary DataFrame
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy Score': [f"{score:.2%}" for score in results.values()]
})

display(comparison_df.sort_values(by='Accuracy Score', ascending=False).reset_index(drop=True))

In [ ]:
## Conclusion & Recommendations

1. **Top Performer:** **Random Forest Classifier** achieved the highest accuracy and F1-score among all models tested.
2. **Key Driver Features:** `alcohol`, `sulphates`, and `volatile acidity` are the most significant chemical properties determining high-quality wines.
3. **Deployment Strategy:** Random Forest is recommended for deployment due to its non-linear modeling capabilities, robustness to outliers, and high overall precision in identifying premium wines.